# Getting Started

Recommendation: work with a conda environment

```sh
conda create -n cnd_hsdvmx_features
conda activate cnd_hsdvmx_features

conda install pytorch-gpu torchvision torchaudio pytorch-cuda=YOUR_VERSION -c pytorch -c nvidia
```

# Functions & Libraries

In [2]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.get_device_name(0)

'NVIDIA GeForce GTX 1650 Ti'

In [3]:
import torch
import torch.nn as nn
import numpy as np
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [4]:
class ModalityEncoder(nn.Module):
    """Apply model LSTM + FC
    By default target_n=64 
    """
    def __init__(self, input_m, hidden_size, output_m, target_n=64):
        super(ModalityEncoder, self).__init__()
        # bidirectional lstm to capture info
        self.lstm = nn.LSTM(input_m, hidden_size, batch_first=True, bidirectional=True)
        self.adaptive_pool = nn.AdaptiveAvgPool1d(target_n)
        self.fc = nn.Linear(hidden_size * 2, output_m)
        
        # trying to preserve the initial semantics
        for name, param in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
    
    def forward(self, x):
        # x shape: (1, N, M_in)
        lstm_out, _ = self.lstm(x)
        # pooling over the temporal dim (N -> 64)
        out = lstm_out.transpose(1, 2)
        out = self.adaptive_pool(out)
        out = out.transpose(1, 2)
        # transforming m dimension
        return self.fc(out)

In [ ]:
def run_modality_pipeline(_modality_type, _input_dir, _output_dir, _m_in, _m_out, _hidden_size, _n_out, _normalize=False):
    """
    Params
    -------
    _modality_type: str; 'audio', 'text' or 'video'
    _input_dir: str; source path directory
    _output_dir: str; path to save the processed files
    _m_in: int; original dimension
    _m_out: int; output dimension after processing
    _hidden_size: int; hidden size of the LSTM
    _n_out: int; output length of the n dimension after processing
    _normalize: bool; whether to normalize the data before processing

    Returns
    -------
    model: the trained ModalityEncoder model
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    input_path = Path(_input_dir)
    print("input_path:", input_path)
    output_path = Path(_output_dir)
    print("output_path:", output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    # 1. instantiate the 'base' model
    model = ModalityEncoder(_m_in, _hidden_size, _m_out, _n_out).to(device)
    model.eval()
    torch.save(model.state_dict(), f'./encoder_{_modality_type}_base.pth')
    
    # 2. files to process
    if _modality_type == 'video':
        search_pattern = "*_rgb.npy" 
         # note: for video use _rgb files to identify the ids
    elif _modality_type == 'audio' or _modality_type == 'text':
        search_pattern = "*.npy"
    else:
        print("ERROR _modality_type")
    files = list(input_path.glob(search_pattern))
    print(f">>> BEGINING {_modality_type.upper()} ({len(files)} files)")

    for f in files:
        if _modality_type == 'video':
            video_id = f.name.replace("_rgb.npy", "") 
            save_name = f"{video_id}.npy"
        else:
            video_id = f.stem
            save_name = f.name

        with torch.no_grad():
            # adapt for each modality
            if _modality_type == 'video':
                rgb = np.load(input_path / f"{video_id}_rgb.npy")
                flow = np.load(input_path / f"{video_id}_flow.npy")
                try:
                    data = np.concatenate([rgb, flow], axis=1) # for video concatenate before process
                except:
                    print(f"{video_id}_rgb.npy")
            else:
                data = np.load(f, allow_pickle=True)

            # optional to normalize the data
            if _normalize:
                data = StandardScaler().fit_transform(data)

            # process the data 
            tensor_in = torch.FloatTensor(data).unsqueeze(0).to(device)
            tensor_out = model(tensor_in)
            
            # save the new processed files
            np.save(output_path / save_name, tensor_out.squeeze(0).cpu().numpy())

    print(f"--- FINISHED: {_modality_type.upper()} ---")
    return model

# Process modalities

BASE_RAW = path from origin

| Modality | pretrained model | N | M_in | NPY_DIR | LSTM+FC | n_out | m_out | npy_dir | 
|----------|------------------|---|---|---------|---------|---|---|---------|
| Audio | vggish | n | 128 | ```npy_audio_vggish``` | ```encoder_audio_base.pth``` |  | 128 | ```data_standardized/audio``` |
| Text | BETO | n | 768 | ```npy_text_beto``` | ```encoder_text_base.pth``` |  | 384 | ```data_standardized/text``` |
| Video | i3D (flow&rgb) | n | 1024+1024* | ```npy_video_i3d``` | ```encoder_video_base.pth``` |  | 1024 | ```data_standardized/video``` |

*Previously concatenated flow + rgb

BASE_OUT = path were the processed modalities would be saved
```data_standardized```

## Audio

In [ ]:
if __name__ == "__main__":
    BASE_RAW = Path("./npy_audio_vggish/")
    BASE_OUT = Path("./data_standardized")

    # 1. PROCESAR AUDIO
    mdl = run_modality_pipeline(
        _modality_type='audio',
        _input_dir=BASE_RAW,
        _output_dir=BASE_OUT / "audio",
        _m_in=128,
        _m_out=128,
        _hidden_size=128,
        _n_out=50
    )

device: cuda
input_path: npy_audio_vggish
output_path: data_standardized/audio
>>> BEGINING AUDIO (1756 files)
--- FINISHED: AUDIO ---


In [7]:
print(mdl)

ModalityEncoder(
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (adaptive_pool): AdaptiveAvgPool1d(output_size=50)
  (fc): Linear(in_features=256, out_features=128, bias=True)
)


## Text

In [ ]:
if __name__ == "__main__":
    BASE_RAW = Path("./npy_text_beto/")
    BASE_OUT = Path("./data_standardized")

    # 2. PROCESAR TEXTO
    mdl = run_modality_pipeline(
        _modality_type='text',
        _input_dir=BASE_RAW,
        _output_dir=BASE_OUT / "text",
        _m_in=768,
        _m_out=768, # reduce (?384
        _hidden_size=512,
        _n_out=50,
        _normalize=True
    )

device: cuda
input_path: npy_text_beto
output_path: data_standardized/text
>>> BEGINING TEXT (1768 files)
--- FINISHED: TEXT ---


In [9]:
print(mdl)

ModalityEncoder(
  (lstm): LSTM(768, 512, batch_first=True, bidirectional=True)
  (adaptive_pool): AdaptiveAvgPool1d(output_size=50)
  (fc): Linear(in_features=1024, out_features=768, bias=True)
)


## Video

In [ ]:
if __name__ == "__main__":
    BASE_RAW = Path("./npy_video_i3d/")
    BASE_OUT = Path("./data_standardized")

    # 3. PROCESAR VIDEO (Fusión Interna)
    mdl = run_modality_pipeline(
        _modality_type='video',
        _input_dir=BASE_RAW,
        _output_dir=BASE_OUT / "video",
        _m_in=2048, # RGB + FLOW (each one has 1024)
        _m_out=2048, # 1024
        _hidden_size=512,
        _n_out=50
    )

print(mdl)

device: cuda
input_path: npy_video_i3d
output_path: data_standardized/video
>>> BEGINING VIDEO (1760 files)
jDVvBm51zG8.04_rgb.npy
--- FINISHED: VIDEO ---
ModalityEncoder(
  (lstm): LSTM(2048, 512, batch_first=True, bidirectional=True)
  (adaptive_pool): AdaptiveAvgPool1d(output_size=50)
  (fc): Linear(in_features=1024, out_features=2048, bias=True)
)


In [11]:
print(mdl)

ModalityEncoder(
  (lstm): LSTM(2048, 512, batch_first=True, bidirectional=True)
  (adaptive_pool): AdaptiveAvgPool1d(output_size=50)
  (fc): Linear(in_features=1024, out_features=2048, bias=True)
)
